# Proj2 — Debug 8-Config RapidFire Sweep

Isolated notebook for debugging the 8-config schema-linking SFT grid from `proj2-scratch.ipynb`.

**Use this notebook when:**
- You hit CUDA / NVML allocator errors during `run_fit`
- You want to run **one run at a time** (set `RUN_IDS = [5]`)
- You want a **fresh experiment name** each session (no `_7` / `_8` suffix churn)

**GPU memory (section 0):** `LOW_VRAM` + `USE_QLORA_15B` shrink batch/seq and load 1.5B in 4-bit. `NUM_CHUNKS=1` when `LOW_VRAM` so each run gets a fresh worker (reduces NVML allocator errors after 0.5B → 1.5B).

**Run catalog (matches scratch notebook grid):**

| Run ID | Model | LR | LoRA |
|--------|-------|-----|------|
| 0 | Qwen2.5-0.5B | 1e-4 | narrow (r=8) |
| 1 | Qwen2.5-0.5B | 1e-4 | wide (r=32) |
| 2 | Qwen2.5-0.5B | 2e-4 | narrow |
| 3 | Qwen2.5-0.5B | 2e-4 | wide |
| 4 | Qwen2.5-1.5B | 1e-4 | narrow |
| 5 | Qwen2.5-1.5B | 1e-4 | wide |
| 6 | Qwen2.5-1.5B | 2e-4 | narrow |
| 7 | Qwen2.5-1.5B | 2e-4 | wide |

**Before starting:** run the path bootstrap cell (section 0), restart kernel if needed, call `experiment.end()` if reusing a session, and ensure `rapidfireai start` is running if you use the UI.

See **`try/README.md`** for teammate setup (clone full repo, no hardcoded `/home/...` paths).

## 0. Debug knobs (edit here)

In [ ]:
from datetime import datetime

from paths import LOGS_DIR, REPO_ROOT, TRAIN_JSON, VAL_JSON, print_paths, require_release_data, setup

setup(chdir_try=True)  # cwd → try/; ./logs → try/logs; imports schema_*.py from here
require_release_data()
print_paths()

# --- What to run ---
RUN_IDS = list(range(8))   # all 8; e.g. [0] or [5] to bisect failures
# RUN_IDS = [5]

# --- Fast debug vs full training ---
DEBUG = True               # True: 1 epoch, smaller batches, shorter seqs
N_TRAIN = 32 if DEBUG else 64
N_VAL = 16 if DEBUG else 32

# --- Experiment naming (timestamp avoids RapidFire _N suffix buildup) ---
EXPERIMENT_BASE = "proj2-debug-8config"
EXPERIMENT_NAME = f"{EXPERIMENT_BASE}-{datetime.now():%Y%m%d-%H%M}"
EXPERIMENT_PATH = "./logs"

# --- GPU memory (DSMLP MIG — avoids NVML / CUDACachingAllocator on 1.5B wide) ---
LOW_VRAM = True                  # per-model batch/seq; always NUM_CHUNKS=1
USE_QLORA_15B = True             # 4-bit load for Qwen 1.5B (~3GB less VRAM)

# --- RapidFire scheduling ---
NUM_CHUNKS = 1 if (DEBUG or LOW_VRAM) else 4   # 1 = fresh worker per run
NUM_GPUS = 1                     # single MIG slice
SEED = 42

SCHEMA_STYLE = "minimal"         # or "full"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"RUN_IDS={RUN_IDS}  DEBUG={DEBUG}  LOW_VRAM={LOW_VRAM}  QLORA_15B={USE_QLORA_15B}")
print(f"N_TRAIN={N_TRAIN}  N_VAL={N_VAL}  NUM_CHUNKS={NUM_CHUNKS}")

REPO_ROOT   = /home/emc001/schema-linker-sft-pipeline
TRY_DIR     = /home/emc001/schema-linker-sft-pipeline/try
SCHEMAS_DIR = /home/emc001/schema-linker-sft-pipeline/schemas  (18 json files)
LOGS_DIR    = /home/emc001/schema-linker-sft-pipeline/try/logs
RF_HOME     = /home/emc001/rapidfireai
cwd         = /home/emc001/schema-linker-sft-pipeline/try
Experiment: proj2-debug-8config-20260528-1754
RUN_IDS=[0, 1, 2, 3, 4, 5, 6, 7]  DEBUG=True  LOW_VRAM=True  QLORA_15B=True
N_TRAIN=32  N_VAL=16  NUM_CHUNKS=1


## 1. Optional — free GPU / end stale experiment

In [4]:
import gc

def cuda_cleanup():
  """Call between failed runs to reduce allocator/NVML pressure."""
  gc.collect()
  try:
    import torch
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.synchronize()
      print("CUDA cache cleared")
  except Exception as e:
    print(f"cuda_cleanup skipped: {e}")

# Uncomment if a previous Experiment object is still in memory:
# experiment.end()
# cuda_cleanup()

## 2. Imports & data

In [5]:
import json

from datasets import Dataset
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from schema_formatting import make_formatting_func
from schema_metrics import compute_metrics_for_rf

# TRAIN_JSON / VAL_JSON set in section 0 via paths.setup()
with open(TRAIN_JSON) as f:
    raw_train = json.load(f)
with open(VAL_JSON) as f:
    raw_val = json.load(f)

train_ds = Dataset.from_list(raw_train[:N_TRAIN])
val_ds = Dataset.from_list(raw_val[:N_VAL])
fmt_func = make_formatting_func(SCHEMA_STYLE)

print(f"Train={len(train_ds)}  Val={len(val_ds)}  schema={SCHEMA_STYLE}")

Train=32  Val=16  schema=minimal


## 3. Model factory & config grid (8 explicit runs)

In [ ]:
def create_model(model_config: dict):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"].copy()
    use_qlora = model_kwargs.pop("use_qlora", False)

    if use_qlora:
        from transformers import BitsAndBytesConfig
        import torch
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_cfg, **model_kwargs
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    return (model, tokenizer)


peft_narrow = RFLoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
peft_wide = RFLoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)


def _is_15b(model_name: str) -> bool:
    return "1.5B" in model_name or "1.5b" in model_name.lower()


def make_sft_args(lr: float, model_name: str, is_wide: bool) -> RFSFTConfig:
    """Per-model batch/seq: 1.5B wide is the heaviest (Run 5 / 7 NVML failures)."""
    is_15b = _is_15b(model_name)
    heavy = is_15b and is_wide

    if DEBUG:
        epochs, log_steps = 1, 5
    else:
        epochs, log_steps = 3, 10

    if LOW_VRAM:
        if heavy:
            train_bs, eval_bs, accum, max_len = 1, 1, 8, 512
        elif is_15b:
            train_bs, eval_bs, accum, max_len = 1, 1, 8, 512
        else:
            train_bs, eval_bs, accum, max_len = 2, 2, 4, 512
    elif DEBUG:
        train_bs, eval_bs, accum, max_len = 2, 2, 2, 512
    else:
        if is_15b:
            train_bs, eval_bs, accum, max_len = 2, 2, 8, 768
        else:
            train_bs, eval_bs, accum, max_len = 4, 4, 4, 1024

    return RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=accum,
        num_train_epochs=epochs,
        max_length=max_len,
        logging_steps=log_steps,
        eval_strategy="epoch",
        load_best_model_at_end=False,
        gradient_checkpointing=is_15b,
        bf16=True,
        fp16=False,
        report_to="none",
    )


MODEL_KWARGS = {"device_map": "auto", "torch_dtype": "auto", "use_cache": False}
GEN_CFG = {"max_new_tokens": 256, "do_sample": False, "temperature": 1.0}
GEN_CFG_15B = {**GEN_CFG, "max_new_tokens": 128}  # less VRAM during eval generation

QWEN_05B = "Qwen/Qwen2.5-0.5B-Instruct"
QWEN_15B = "Qwen/Qwen2.5-1.5B-Instruct"

# One RFModelConfig per run (no nested peft List → easier to subset by RUN_IDS)
# Order: 0.5B first, then 1.5B narrow, then 1.5B wide (heaviest last).
RUN_CATALOG = [
    {"run_id": 0, "model": QWEN_05B, "lr": 1e-4, "peft": peft_narrow, "wide": False, "label": "0.5B | lr=1e-4 | narrow"},
    {"run_id": 1, "model": QWEN_05B, "lr": 1e-4, "peft": peft_wide,   "wide": True,  "label": "0.5B | lr=1e-4 | wide"},
    {"run_id": 2, "model": QWEN_05B, "lr": 2e-4, "peft": peft_narrow, "wide": False, "label": "0.5B | lr=2e-4 | narrow"},
    {"run_id": 3, "model": QWEN_05B, "lr": 2e-4, "peft": peft_wide,   "wide": True,  "label": "0.5B | lr=2e-4 | wide"},
    {"run_id": 4, "model": QWEN_15B, "lr": 1e-4, "peft": peft_narrow, "wide": False, "label": "1.5B | lr=1e-4 | narrow"},
    {"run_id": 6, "model": QWEN_15B, "lr": 2e-4, "peft": peft_narrow, "wide": False, "label": "1.5B | lr=2e-4 | narrow"},
    {"run_id": 5, "model": QWEN_15B, "lr": 1e-4, "peft": peft_wide,   "wide": True,  "label": "1.5B | lr=1e-4 | wide"},
    {"run_id": 7, "model": QWEN_15B, "lr": 2e-4, "peft": peft_wide,   "wide": True,  "label": "1.5B | lr=2e-4 | wide"},
]


def catalog_entry_to_rf_config(entry: dict) -> RFModelConfig:
    is_15b = _is_15b(entry["model"])
    model_kwargs = MODEL_KWARGS.copy()
    if is_15b and USE_QLORA_15B:
        model_kwargs["use_qlora"] = True

    gen_cfg = GEN_CFG_15B if is_15b else GEN_CFG

    return RFModelConfig(
        model_name=entry["model"],
        peft_config=entry["peft"],
        training_args=make_sft_args(entry["lr"], entry["model"], entry["wide"]),
        model_type="causal_lm",
        model_kwargs=model_kwargs,
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config=gen_cfg.copy(),
    )


selected = [e for e in RUN_CATALOG if e["run_id"] in RUN_IDS]
if not selected:
    raise ValueError(f"RUN_IDS {RUN_IDS} matched no runs")

for e in selected:
    qlora = " qlora" if _is_15b(e["model"]) and USE_QLORA_15B else ""
    print(f"  run {e['run_id']}: {e['label']}{qlora}")

config_set = List([catalog_entry_to_rf_config(e) for e in selected])
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

print(f"\nLaunching {len(selected)} run(s) (RapidFire will assign run_id 0..{len(selected)-1} within this experiment)")

  run 0: 0.5B | lr=1e-4 | narrow
  run 1: 0.5B | lr=1e-4 | wide
  run 2: 0.5B | lr=2e-4 | narrow
  run 3: 0.5B | lr=2e-4 | wide
  run 4: 1.5B | lr=1e-4 | narrow qlora
  run 6: 1.5B | lr=2e-4 | narrow qlora
  run 5: 1.5B | lr=1e-4 | wide qlora
  run 7: 1.5B | lr=2e-4 | wide qlora

Launching 8 run(s) (RapidFire will assign run_id 0..7 within this experiment)


## 4. Create experiment & run

In [7]:
cuda_cleanup()

experiment = Experiment(
    experiment_name=EXPERIMENT_NAME,
    mode="fit",
    experiment_path=EXPERIMENT_PATH,
)
print(f"Logs: {EXPERIMENT_PATH}/{experiment.experiment_name}")

CUDA cache cleared
Experiment proj2-debug-8config-20260528-1754 created with Experiment ID: 29 and Metric Experiment ID: 47 at /home/emc001/schema-linker-sft-pipeline/try/logs/proj2-debug-8config-20260528-1754
Logs: ./logs/proj2-debug-8config-20260528-1754


In [ ]:
# Blocking run_fit — LOW_VRAM + NUM_CHUNKS=1 reduce NVML allocator failures on 1.5B wide.
# If a run still fails: RUN_IDS=[5], kernel restart, experiment.end(), cuda_cleanup(), retry.
experiment.run_fit(
    config_group,
    create_model,
    train_ds,
    val_ds,
    num_chunks=NUM_CHUNKS,
    seed=SEED,
    num_gpus=NUM_GPUS,
)

INFO 05-28 17:55:36 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-28 17:55:50 [__init__.py:216] Automatically detected platform cuda.


## 5. Results & cleanup

In [ ]:
results = experiment.get_results()
display(results)

In [ ]:
experiment.end()
cuda_cleanup()
print("Experiment ended.")